# 10 Machine Learning — Exercises

Practice sklearn classification pipelines with the Legionnaires' disease data from Pine and Cypress Nursing Home.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.inspection import permutation_importance

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["severe_outcome"] = ((df["hospitalized"] == 1) | (df["outcome"] == "dead")).astype(int)

## Question 1: The effect of class_weight="balanced"

1. Build a Pipeline (ColumnTransformer + LogisticRegression)
2. Train it separately with `class_weight=None` and `class_weight="balanced"`
3. Compare the two on Task B (severe_outcome) using 5-fold CV AUC
4. Which one is better? Why?

In [ ]:
# TODO: define features and preprocessing
# TODO: Pipeline with class_weight=None
# TODO: Pipeline with class_weight="balanced"
# TODO: compare 5-fold CV AUC

## Question 2: Feature importance for Task B (severe outcome prediction)

1. Run 5-fold CV AUC with a Random Forest on Task B (severe_outcome)
2. Use permutation importance to find the top 5 most important features
3. Draw a horizontal bar chart of the feature importances
4. How does the ranking differ from the important features for Task A (infected)?

In [ ]:
# TODO: Random Forest pipeline
# TODO: 5-fold CV AUC
# TODO: permutation importance
# TODO: barh plot + top 5

## Question 3 (challenge): Three-model comparison + ROC curves

1. Build three Pipelines: Logistic Regression, Random Forest, Gradient Boosting
2. Train and test with a 70/30 split (`random_state=42`)
3. Compute each model's test AUC on Task A
4. Plot all three ROC curves on the same figure
5. Which model performs best? Is a conclusion from 280 rows reliable?

In [ ]:
# TODO: three Pipelines
# TODO: train_test_split(test_size=0.3, random_state=42)
# TODO: AUC for each model
# TODO: ROC curves (from sklearn.metrics import roc_curve)
# TODO: interpretation

## Question 4: COVID-19 severe illness prediction (COVID-19 scenario)

Build a binary classification model for COVID-19 severe illness using demographic and comorbidity data.

1. Use `age/male/diabetes/hypertension/vaccinated` as features and `severe` as the label
2. Split into train/test sets, and build a `StandardScaler + LogisticRegression` Pipeline
3. Compute accuracy, ROC-AUC, and the confusion matrix on the test set
4. Interpret the direction of each feature from the standardized coefficients (especially vaccination)

In [ ]:
# COVID-19: predict "severe" illness from demographics and comorbidities (binary classification)
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
rng = np.random.default_rng(1004)
n = 1200
age = rng.integers(20, 90, n)
male = rng.integers(0, 2, n)
diabetes = rng.integers(0, 2, n)
hypertension = rng.integers(0, 2, n)
vaccinated = rng.binomial(1, 0.6, n)
logit = (-6 + 0.06 * age + 0.4 * male + 0.7 * diabetes + 0.5 * hypertension - 1.2 * vaccinated)
severe = rng.binomial(1, 1 / (1 + np.exp(-logit)))
covid = pd.DataFrame({"age": age, "male": male, "diabetes": diabetes,
                      "hypertension": hypertension, "vaccinated": vaccinated, "severe": severe})
print(covid["severe"].value_counts(normalize=True).round(3).to_dict())

# TODO: use age/male/diabetes/hypertension/vaccinated as features, severe as the label
# TODO: train_test_split (stratify=y, test_size=0.25, random_state=42)
# TODO: build a Pipeline(StandardScaler + LogisticRegression) and fit it
# TODO: compute accuracy, ROC-AUC, confusion_matrix on the test set
# TODO: interpret which features raise severe-illness risk, and the direction for vaccination

## Question 5: Predicting severe dengue (DHF) (dengue scenario)

Predict whether dengue fever progresses to severe dengue (dengue hemorrhagic fever, DHF).

1. Features `age/secondary_infection/platelet/days_fever`, label `dhf`
2. Train a `RandomForestClassifier`, and compute the test-set ROC-AUC and confusion matrix
3. Inspect `feature_importances_` to identify the most critical risk factor (hint: ADE)

In [ ]:
# Dengue: predict progression to severe dengue (DHF) (random forest)
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
rng = np.random.default_rng(1005)
n = 1000
age = rng.integers(1, 80, n)
secondary_infection = rng.binomial(1, 0.45, n)   # secondary infection is an ADE risk
platelet = rng.normal(180, 60, n).clip(20, 400)  # platelet count (thousand/uL)
days_fever = rng.integers(1, 8, n)
logit = (-2.5 + 1.6 * secondary_infection - 0.012 * platelet + 0.15 * days_fever + 0.01 * age)
dhf = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dengue = pd.DataFrame({"age": age, "secondary_infection": secondary_infection,
                       "platelet": platelet.round(0), "days_fever": days_fever, "dhf": dhf})
print(f"DHF severe illness rate: {dengue['dhf'].mean():.1%}")

# TODO: features = age/secondary_infection/platelet/days_fever, label = dhf
# TODO: after train_test_split, use RandomForestClassifier (n_estimators=200)
# TODO: compute test-set ROC-AUC and confusion matrix
# TODO: check feature_importances_ to determine which factor matters most (hint: secondary infection / ADE)

## Question 6: Predicting tuberculosis treatment outcome (TB scenario)

Predict whether a TB patient's treatment succeeds, and compare two models.

1. Features `age/mdr/hiv/adherence`, label `success`
2. Compare LogisticRegression and RandomForest using `cross_val_score(cv=5, scoring='roc_auc')`
3. Interpret the effect of medication adherence (`adherence`) on treatment success

In [ ]:
# TB: predict treatment outcome (success vs failure/loss to follow-up), compare two models
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1006)
n = 900
age = rng.integers(18, 85, n)
mdr = rng.binomial(1, 0.15, n)          # drug resistance
hiv = rng.binomial(1, 0.1, n)
adherence = rng.uniform(0.4, 1.0, n)    # medication adherence
logit = (2.0 - 1.8 * mdr - 1.2 * hiv + 3.0 * (adherence - 0.7) - 0.01 * age)
success = rng.binomial(1, 1 / (1 + np.exp(-logit)))
tb = pd.DataFrame({"age": age, "mdr": mdr, "hiv": hiv,
                   "adherence": adherence.round(2), "success": success})
print(f"Treatment success rate: {tb['success'].mean():.1%}")

# TODO: features = age/mdr/hiv/adherence, label = success
# TODO: compare LogisticRegression and RandomForest with cross_val_score (cv=5, scoring="roc_auc")
# TODO: print the mean AUC for both models and decide which is better
# TODO: interpret the effect of adherence (medication adherence) on treatment success

## Question 7: Predicting influenza hospitalization and plotting an ROC curve (influenza scenario)

Predict whether an influenza patient is hospitalized.

1. Features `age/chronic/vaccinated/onset_to_care`, label `hospitalized`
2. Train a `GradientBoostingClassifier` and compute the ROC-AUC
3. Use `roc_curve` to plot the ROC curve, and interpret the direction of the vaccination effect

In [ ]:
# Influenza: predict hospitalization (gradient boosting + ROC curve)
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1007)
n = 1100
age = rng.integers(0, 95, n)
chronic = rng.binomial(1, 0.25, n)
vaccinated = rng.binomial(1, 0.5, n)
onset_to_care = rng.integers(0, 6, n)  # days from symptom onset to seeking care
logit = (-3.5 + 0.05 * age + 1.0 * chronic - 0.8 * vaccinated + 0.25 * onset_to_care)
hosp = rng.binomial(1, 1 / (1 + np.exp(-logit)))
flu = pd.DataFrame({"age": age, "chronic": chronic, "vaccinated": vaccinated,
                    "onset_to_care": onset_to_care, "hospitalized": hosp})
print(f"Hospitalization rate: {flu['hospitalized'].mean():.1%}")

# TODO: features = age/chronic/vaccinated/onset_to_care, label = hospitalized
# TODO: train a GradientBoostingClassifier, compute test-set ROC-AUC
# TODO: use roc_curve to plot the ROC curve
# TODO: interpret the direction of vaccination's effect on hospitalization risk

## Question 8 (challenge): Sepsis ICU prognosis and feature importance (sepsis scenario)

Predict in-hospital death for sepsis patients, and compare two kinds of feature importance.

1. Features `age/lactate/sofa/wbc/comorbid`, label `death`
2. Train a `RandomForestClassifier` and compute the ROC-AUC
3. Rank feature importance with `permutation_importance` (on the test set)
4. Interpret: which bedside indicators matter most? How does `permutation_importance` differ from `feature_importances_`, and why is the former more trustworthy?

In [ ]:
# Sepsis: ICU prognosis prediction + feature importance (challenge)
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(1008)
n = 1000
age = rng.integers(18, 95, n)
lactate = rng.normal(2.5, 1.5, n).clip(0.5, 12)   # lactate
sofa = rng.integers(0, 18, n)                       # SOFA score
wbc = rng.normal(12, 6, n).clip(1, 40)
comorbid = rng.integers(0, 4, n)
logit = (-4 + 0.03 * age + 0.45 * lactate + 0.25 * sofa + 0.3 * comorbid + 0.01 * wbc)
death = rng.binomial(1, 1 / (1 + np.exp(-logit)))
sepsis = pd.DataFrame({"age": age, "lactate": lactate.round(1), "sofa": sofa,
                       "wbc": wbc.round(1), "comorbid": comorbid, "death": death})
print(f"In-hospital death rate: {sepsis['death'].mean():.1%}")

# TODO: features = age/lactate/sofa/wbc/comorbid, label = death
# TODO: after train_test_split, train a RandomForestClassifier and compute ROC-AUC
# TODO: use permutation_importance (on the test set) to rank feature importance
# TODO: interpret -- which bedside indicators contribute most to predicting death?
#        How does permutation_importance differ from feature_importances_, and
#        why is the former more trustworthy?